<a href="https://colab.research.google.com/github/Rahul2004Yadav/rahul-repo/blob/main/e_doc_LORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install unsloth # install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.6/275.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-5cg4yz8i
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-5cg4yz8i
  Resolved https://github.com/unslothai/unsloth.git to commit c5a2a36e47e4179eca7d2557063a5a3e82611b36
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.5.10-py3-none-any.whl size=276116 sha256=91ab05c9ad4c62f1df196dfe2d4b5639e515a3d6cd83063fc779ef3206e4a761
  Stored in directory: /tmp/pip-ephem-wheel-cache-amzxg0wz/wheels/d1/17/05/850ab10c33284a4763b0595cd8ea9d01fce6e221cac24b3c01
Successfully built unsloth
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.5.9
    Uninstalling unsloth-2025.5.9:
      Successfully uninstalled unsloth-2025.5.9


In [ ]:
# Step3: Import necessary libraries
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(hf_token)


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU device: Tesla T4


In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_sequence_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token
)

==((====))==  Unsloth 2025.5.10: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
prompt_style= """
You are a conversational AI assistant acting as a professionally worded “e-doctor.” Your role is to provide preliminary, text-based guidance in response to patient symptom-related questions. You must **not** substitute for a licensed medical professional.

Before responding, analyze the user’s input carefully. Follow a structured reasoning process and ensure the output:

- Adheres to established clinical guidelines or best practices.
- Is phrased in clear, formal, and empathetic medical language.
- Includes appropriate disclaimers indicating the informational nature of the advice.

### Objective:
Healthcare providers are adopting AI-assisted tools to support triage and early guidance. This system is designed to simulate a responsible AI health assistant trained on a medically-relevant dataset using decoder-only transformer models. It compares three fine-tuning approaches for training: Prompt Tuning, LoRA/QLoRA with PEFT, and Full Fine-Tuning.

### Task:
Respond to the medical inquiry below by generating a high-quality, medically sound answer that balances informativeness and caution.

### Input (Patient Query):
{}

### Output (AI Assistant Response):
{}

*Disclaimer: This response is generated for informational purposes only and should not be considered a substitute for professional medical advice, diagnosis, or treatment. Please consult a licensed healthcare provider for personalized care.*
"""


In [ ]:
question = """A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or
              sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
              what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

FastLanguageModel.for_inference(model)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs)


print(response)

["<｜begin▁of▁sentence｜>\nYou are a conversational AI assistant acting as a professionally worded “e-doctor.” Your role is to provide preliminary, text-based guidance in response to patient symptom-related questions. You must **not** substitute for a licensed medical professional.\n\nBefore responding, analyze the user’s input carefully. Follow a structured reasoning process and ensure the output:\n\n- Adheres to established clinical guidelines or best practices.\n- Is phrased in clear, formal, and empathetic medical language.\n- Includes appropriate disclaimers indicating the informational nature of the advice.\n\n### Objective:\nHealthcare providers are adopting AI-assisted tools to support triage and early guidance. This system is designed to simulate a responsible AI health assistant trained on a medically-relevant dataset using decoder-only transformer models. It compares three fine-tuning approaches for training: Prompt Tuning, LoRA/QLoRA with PEFT, and Full Fine-Tuning.\n\n### Ta

In [ ]:
print(response[0].split("### Output (AI Assistant Response):")[1])




*Disclaimer: This response is generated for informational purposes only and should not be considered a substitute for professional medical advice, diagnosis, or treatment. Please consult a licensed healthcare provider for personalized care.*
</think>

**Response:**

I'm sorry to hear about your symptoms. Based on the information provided, it seems you're experiencing involuntary urine loss during activities like coughing or sneezing, but not at night. This is often referred to as stress urinary incontinence. 

The gynecological exam and Q-tip test have been conducted, and now the question is about what cystometry (a specific test to evaluate bladder function) might reveal regarding your residual volume and detrusor contractions.

Cystometry is a diagnostic procedure that assesses how your bladder behaves when it's filled and emptied. It can show whether your bladder is overactive (detrusor contractions) or if there's residual volume in the bladder that isn't emptied completely after

In [ ]:
import json
from datasets import Dataset


with open("/content/combined_output2_final.json", "r") as f:
    raw_data = json.load(f)

# Build input-output pairs
formatted_data = []
for entry in raw_data:
    utterances = entry.get("utterances", [])
    if len(utterances) >= 2:
        patient_utterance = utterances[0].replace("patient:", "").strip()
        doctor_response = utterances[1].replace("doctor:", "").strip()

        prompt_style = f"""Given the following patient query, provide a helpful and informative response from the perspective of a doctor.\n\nPatient query: {patient_utterance}\n\nYour response:"""

        formatted_data.append({
            "input": prompt_style,
            "output": doctor_response
        })

# Create Hugging Face dataset
hf_dataset = Dataset.from_list(formatted_data)


In [ ]:
hf_dataset[10]

{'input': 'Given the following patient query, provide a helpful and informative response from the perspective of a doctor.\n\nPatient query: should i really be worried about my elderly parents catching covid-19? what steps should they take to stay healthy?\n\nYour response:',
 'output': 'yes. avoid contact but ensure that they have food, supplies and able to slay at home. they should drink fluids and if they develop a fever, please call your pcp right away and also consider a virtual appointment with ht. other symptoms to watch for include dry cough and shortness of breath. use good hand washing and disinfect surfaces.'}

In [ ]:
EOS_TOKEN = tokenizer.eos_token  # Define EOS_TOKEN which tells the model when to stop generating text during training
EOS_TOKEN

'<｜end▁of▁sentence｜>'

In [ ]:
train_prompt_style = """Below is a task designed to simulate a responsible, medically informed AI assistant ("e-doctor") that offers preliminary guidance for common patient inquiries. The AI assistant is trained to follow recognized clinical guidelines, use professional language, and always clarify that it does not replace a licensed healthcare provider.

### Instruction:
You are a virtual medical assistant providing preliminary, text-based health advice. Analyze the question carefully, think through the relevant clinical reasoning, and craft a formal response that is informative, safe, and contextually appropriate.

### Patient Query:
{}

### AI Assistant Response:
{}

*Disclaimer: This response is generated for informational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment. Please consult a licensed healthcare provider for personalized care.*
"""


In [ ]:
hf_dataset.features

{'input': Value(dtype='string', id=None),
 'output': Value(dtype='string', id=None)}

In [ ]:
train_prompt_style = """Below is a task designed to simulate a responsible, medically informed AI assistant ("e-doctor") that offers preliminary guidance for common patient inquiries. The AI assistant is trained to follow recognized clinical guidelines, use professional language, and always clarify that it does not replace a licensed healthcare provider.

### Instruction:
You are a virtual medical assistant providing preliminary, text-based health advice. Analyze the question carefully, think through the relevant clinical reasoning, and craft a formal response that is informative, safe, and contextually appropriate.

### Patient Query:
{}

### AI Assistant Response:
{}

*Disclaimer: This response is generated for informational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment. Please consult a licensed healthcare provider for personalized care.*
"""

# EOS_TOKEN = "</s>"


def preprocess_input_data(example):
    inputs = example["input"]
    outputs = example["output"]

    # Extract patient query (assuming it's after "Patient query:")
    if "Patient query:" in inputs:
        patient_query = inputs.split("Patient query:")[-1].strip()
    else:
        patient_query = inputs.strip()

    # Format into training text
    texts = train_prompt_style.format(patient_query, outputs) + EOS_TOKEN

    return {
        "text": texts
    }


finetune_dataset = hf_dataset.map(preprocess_input_data)


Map:   0%|          | 0/259766 [00:00<?, ? examples/s]

In [ ]:


finetune_dataset["text"][0]

'Below is a task designed to simulate a responsible, medically informed AI assistant ("e-doctor") that offers preliminary guidance for common patient inquiries. The AI assistant is trained to follow recognized clinical guidelines, use professional language, and always clarify that it does not replace a licensed healthcare provider.\n\n### Instruction:\nYou are a virtual medical assistant providing preliminary, text-based health advice. Analyze the question carefully, think through the relevant clinical reasoning, and craft a formal response that is informative, safe, and contextually appropriate.\n\n### Patient Query:\ngood day. this morning i coughed for the very first time in a long time. with the corona virus around i feel the need to report this. i coughed for about 5 min. i have no fever, not tired and chest feels weird. what should i do?\n\nYour response:\n\n### AI Assistant Response:\nin brief: best to stay home right now stay home, consult here. disinfect everything and stay sa

In [ ]:
from unsloth import FastLanguageModel

# Apply LoRA to your decoder-only model
model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 16,  # LoRA rank
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3047,
    use_rslora = False,
    loftq_config = None
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.5.10 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
if hasattr(model, '_unwrapped_old_generate'):
    del model._unwrapped_old_generate

In [ ]:
trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset,
    dataset_text_field = "texts",
    max_seq_length = 1024,
    dataset_num_proc = 1,

    # Define training args
    args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",


    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/259766 [00:00<?, ? examples/s]

In [ ]:
#  Start fine-tuning
trainer.train()
# trainer.model.save_pretrained("e_doctor_lora")
# tokenizer.save_pretrained("e_doctor_lora")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 259,766 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rahul9664171057 (rahul9664171057-iit-roorkee) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.856500
20,1.805400
30,1.796500
40,1.684500
50,1.697300
60,1.665100


TrainOutput(global_step=60, training_loss=1.9175439834594727, metrics={'train_runtime': 1154.4139, 'train_samples_per_second': 0.416, 'train_steps_per_second': 0.052, 'total_flos': 8111358323122176.0, 'train_loss': 1.9175439834594727})

In [ ]:
wandb.finish()

train/epoch,▁▂▄▅▇██
train/global_step,▁▂▄▅▇██
train/grad_norm,█▁▂▁▃▅
train/learning_rate,█▇▅▄▂▁
train/loss,█▂▂▁▁▁
total_flos,8111358323122176.0
train/epoch,0.00185
train/global_step,60
train/grad_norm,0.90878
train/learning_rate,0.0
train/loss,1.6651


In [ ]:
# Save only the LoRA adapter weights (small & fast)
trainer.model.save_pretrained("e_doctor_lora")
tokenizer.save_pretrained("e_doctor_lora")


('e_doctor_lora/tokenizer_config.json',
 'e_doctor_lora/special_tokens_map.json',
 'e_doctor_lora/chat_template.jinja',
 'e_doctor_lora/tokenizer.json')

In [ ]:
!zip -r e_doctor_lora.zip e_doctor_lora


  adding: e_doctor_lora/ (stored 0%)
  adding: e_doctor_lora/special_tokens_map.json (deflated 69%)
  adding: e_doctor_lora/adapter_model.safetensors (deflated 8%)
  adding: e_doctor_lora/chat_template.jinja (deflated 75%)
  adding: e_doctor_lora/tokenizer_config.json (deflated 96%)
  adding: e_doctor_lora/adapter_config.json (deflated 56%)
  adding: e_doctor_lora/tokenizer.json (deflated 85%)
  adding: e_doctor_lora/README.md (deflated 66%)


In [ ]:
from google.colab import files
files.download("e_doctor_lora.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>